## NB_lab_test_trends
Builds `<catalog>.gold.lab_test_trends` from the silver layer, following STTM `sttm_silver_to_gold.xlsx` (sheet: lab_test_trends).

* **Grain:** one row per test_name per calendar month
* **Driving table:** `silver.lab_results` grouped by test_name, year(test_date), month(test_date)
* **Joins:** LEFT JOIN `silver.calendar` on date_key = yyyyMMdd(first day of month) for month_name / quarter
* **Window:** LAG(abnormal_rate_pct) partitioned by test_name ordered by year, month
* **Load type:** FULL - `DataFrame.writeTo(...).createOrReplace()` (PySpark equivalent of CREATE OR REPLACE TABLE ... AS SELECT)
* **Note:** `silver.calendar` has no `quarter_name` column, so it is derived as `'Q' || calendar.quarter` (e.g. "Q1")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

dbutils.widgets.text("catalog", "dbw_caresync")
catalog = dbutils.widgets.get("catalog").strip() or "dbw_caresync"

source_schema = f"{catalog}.silver"
target_table = f"{catalog}.gold.lab_test_trends"
print(f"[SOURCE SCHEMA] {source_schema}")
print(f"[TARGET TABLE] {target_table}")

In [0]:
print("[STEP 1] Reading silver sources")
lab_results_df = spark.table(f"{source_schema}.lab_results")
calendar_df = spark.table(f"{source_schema}.calendar").alias("cal")

# Exact decimal 100.0 (same as the SQL literal) so rounding of the rate columns is deterministic
HUNDRED = F.lit(100.0).cast("decimal(4,1)")


def status_flag(status):
    """1 when result_status equals the given value, else 0 (used for conditional counts)."""
    return F.when(F.col("result_status") == status, 1).otherwise(0)


print("[STEP 2] Aggregating lab_results per test_name per month")
monthly_df = lab_results_df.groupBy(
    "test_name",
    F.year("test_date").alias("year"),
    F.month("test_date").alias("month"),
).agg(
    F.count("*").alias("total_tests"),
    F.sum(status_flag("Abnormal")).alias("abnormal_test_count"),
    F.sum(status_flag("Critical")).alias("critical_test_count"),
)

with_rates_df = (
    monthly_df
    .withColumn("abnormal_rate_pct", (F.col("abnormal_test_count") * HUNDRED / F.col("total_tests")).cast("decimal(5,2)"))
    .withColumn("critical_rate_pct", (F.col("critical_test_count") * HUNDRED / F.col("total_tests")).cast("decimal(5,2)"))
)

print("[STEP 3] Month-over-month trend: LAG(abnormal_rate_pct) per test_name ordered by year, month")
test_window = Window.partitionBy("test_name").orderBy("year", "month")
with_lag_df = with_rates_df.withColumn(
    "prev_month_abnormal_rate_pct", F.lag("abnormal_rate_pct").over(test_window)
).alias("w")

print("[STEP 4] Joining calendar (LEFT, first day of month) and applying column mapping")
month_start_key = F.date_format(F.make_date(F.col("w.year"), F.col("w.month"), F.lit(1)), "yyyyMMdd").cast("int")

lab_test_trends_df = (
    with_lag_df
    .join(calendar_df, F.col("cal.date_key") == month_start_key, "left")
    .select(
        F.col("w.test_name").cast("string").alias("test_name"),
        F.col("w.year").cast("int").alias("year"),
        F.col("w.month").cast("int").alias("month"),
        F.col("cal.month_name").cast("string").alias("month_name"),
        # silver.calendar has no quarter_name column -> derived from quarter
        F.when(F.col("cal.quarter").isNotNull(), F.concat(F.lit("Q"), F.col("cal.quarter").cast("string")))
        .cast("string")
        .alias("quarter_name"),
        F.col("w.total_tests").cast("int").alias("total_tests"),
        F.col("w.abnormal_test_count").cast("int").alias("abnormal_test_count"),
        F.col("w.critical_test_count").cast("int").alias("critical_test_count"),
        F.col("w.abnormal_rate_pct").alias("abnormal_rate_pct"),
        F.col("w.critical_rate_pct").alias("critical_rate_pct"),
        F.col("w.prev_month_abnormal_rate_pct").cast("decimal(5,2)").alias("prev_month_abnormal_rate_pct"),
        (F.col("w.abnormal_rate_pct") - F.col("w.prev_month_abnormal_rate_pct")).cast("decimal(6,2)").alias("abnormal_rate_change_pct"),
    )
)

print(f"[STEP 5] Writing {target_table} (full rebuild)")
lab_test_trends_df.writeTo(target_table).using("delta").createOrReplace()
spark.sql(
    f"COMMENT ON TABLE {target_table} IS "
    "'Gold: one row per lab test type per month with abnormal/critical rates and month-over-month trend. Built per STTM silver-to-gold v1.0.'"
)
print(f"[STEP 5 COMPLETE] {target_table} rebuilt")

In [0]:
print("[STEP 6] Validating grain and reconciliation")
stats = (
    spark.table(target_table)
    .agg(
        F.count("*").alias("row_count"),
        F.countDistinct("test_name", "year", "month").alias("key_count"),
        F.sum("total_tests").alias("test_count"),
    )
    .first()
)
src_tests = spark.table(f"{source_schema}.lab_results").count()
print(f"[TARGET ROWS] {stats.row_count} | [DISTINCT KEYS] {stats.key_count} | [SUM total_tests] {stats.test_count} | [SOURCE lab_results] {src_tests}")

if stats.row_count != stats.key_count:
    raise ValueError("Grain violation: duplicate (test_name, year, month)")
if stats.test_count != src_tests:
    raise ValueError("Reconciliation failed: SUM(total_tests) != source lab_results count")

print("[STEP 6 COMPLETE] Validation passed")

print("[STEP 7] Computing processed record count")
record_count = lab_test_trends_df.count()
print(f"[STEP 7 COMPLETE] Processed record count: {record_count}")
dbutils.notebook.exit(str(record_count))